In [2]:
import csv
import json
from datetime import datetime
from pathlib import Path

import pandas as pd

In [3]:
base_dir = Path(".").resolve()
data_dir = base_dir / "data"

In [3]:
# Credential Group
with open(data_dir / "credential_group.json") as file:
    data = json.load(file)

df = pd.DataFrame(data.get("clientGroupList", []))
df.rename(columns={"clientCodes": "credential_list"}, inplace=True)
df["id"] = df["id"].astype(int)

df.to_csv(data_dir / "credential_group.csv", index=False)

In [8]:
with open(data_dir / "distribution_rules.json") as file:
    data = json.load(file)

for row in data.get("rules", []):
    print(json.dumps(row, indent=4))
    break

{
    "editState": 1,
    "isObsolete": false,
    "name": "Abades El MIrador",
    "description": "CDH-271194",
    "lastUser": "maria.ulloa@traveltino.com",
    "lastDate": "27-04-2023 17:52:11",
    "tag": 1,
    "id": "394291",
    "lvl": {
        "rrg": {
            "t": 0,
            "f": 0,
            "u": 0
        },
        "cli": {
            "t": 1,
            "l": [
                "5226",
                "8550"
            ]
        },
        "ps": {
            "t": 0,
            "l": null
        },
        "cp": 0,
        "rat": 0,
        "prv": {
            "t": 1,
            "l": [
                "SMD",
                "BCONG"
            ]
        },
        "hot": {
            "t": 1,
            "l": [
                "1727"
            ]
        },
        "dest": {
            "t": 0,
            "l": []
        },
        "mrk": {
            "t": 0,
            "l": null
        },
        "mel": {
            "t": 0,
            "l": null
      

In [15]:
# Distribution Rules
with open(data_dir / "metadata.json") as meta_file:
    metadata = json.load(meta_file)

with open(data_dir / "distribution_rules.json") as file:
    data = json.load(file, object_hook=lambda obj: {**obj.pop("lvl", {}), **obj})
    rules = data.get("rules", [])

    for rule in rules:
        rrg = rule.get("rrg")
        if rrg is None:
            rule["rrg"] = {}

        for key in list(rule.keys()):
            if key in metadata:
                rule[metadata[key]] = rule.pop(key)

        level_mapping = metadata["level_mapping"]
        for key, value in rule.items():
            if isinstance(value, dict):
                for k in list(value.keys()):
                    if k in level_mapping:
                        value[level_mapping[k]] = value.pop(k)

df = pd.json_normalize(rules, sep="_")
df.to_csv(data_dir / "distribution_rules.csv", index=False)

In [36]:
with open(data_dir / "distribution_rules.json") as file:
    data = json.load(file)

rules = data.get("rules", [])

for rule in rules:
    rrg = rule.get("lvl", {}).get("rrg")
    if rrg is None:
        rule["lvl"]["rrg"] = {"t": 0, "f": 0, "u": 0}

    last_date_str = rule.get("lastDate")
    if last_date_str:
        # Convert the existing date format
        last_date = datetime.strptime(last_date_str, "%d-%m-%Y %H:%M:%S")
        # Update the format to 'YYYY-MM-DD HH:MM:SS'
        rule["lastDate"] = last_date.strftime("%Y-%m-%d %H:%M:%S")

df = pd.DataFrame(rules)

In [38]:
df["lastDate"] = pd.to_datetime(df["lastDate"], format="%d-%m-%Y %H:%M:%S")

In [40]:
df["lvl"] = df["lvl"].apply(lambda x: json.dumps(x).replace("'", '"'))

In [46]:
df = df[
    [
        "id",
        "editState",
        "isObsolete",
        "name",
        "description",
        "lastUser",
        "lastDate",
        "tag",
        "lvl",
    ]
]

In [48]:
with pd.option_context("display.max_columns", None):
    display(df.head())

,id,editState,isObsolete,name,description,lastUser,lastDate,tag,lvl
0,394291,1,False,Abades El MIrador,CDH-271194,maria.ulloa@traveltino.com,2023-04-27 17:52:11,1,"{""rrg"": {""t"": 0, ""f"": 0, ""u"": 0}, ""cli"": {""t"":..."
1,432673,1,False,Cierre todo SMY Sheraton FUE,CDH-12945,maria.ulloa@traveltino.com,2024-04-09 14:01:56,1,"{""rrg"": {""t"": 0, ""f"": 0, ""u"": 0}, ""cli"": {""t"":..."
2,498300,1,False,NRF next 15 dias,Stop sales for NRF 15 days in adv to the futur...,daniel.diez@smyrooms.com,2024-07-17 11:38:14,1,"{""rrg"": {""t"": 1, ""f"": 15, ""u"": 9999}, ""cli"": {..."
3,394034,1,False,Close Landmar Costa Gigantes to Atrapalo,Close Landmar Costa Gigantes to Atrapalo,pau.cespedes@logitravelgroup.com,2022-09-21 11:46:13,3,"{""rrg"": {""t"": 0, ""f"": 0, ""u"": 0}, ""cli"": {""t"":..."
4,394446,1,False,Cierre de ventas en el hotel Abba Berlin,CDH-274118,miguel.fuentes@traveltino.com,2023-01-25 09:53:55,1,"{""rrg"": {""t"": 0, ""f"": 0, ""u"": 0}, ""cli"": {""t"":..."


In [10]:
import json

import psycopg

db_params = {
    "dbname": "sandbox",
    "user": "postgres",
    "password": "j7#tS#XQZxtL",
    "host": "129.151.159.251", 
    "port": "5432",
}

In [11]:
def insert_data(rules):
    with psycopg.connect(**db_params) as conn:
        with conn.cursor() as cursor:
            for rule in rules:
                insert_query = """
                INSERT INTO distribution_rule (id, edit_state, obsolete, name, description, last_user, last_date, tag, level)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s);
                """
                cursor.execute(insert_query, (
                    rule.get("id"),
                    rule.get("editState"),
                    rule.get("isObsolete"),
                    rule.get("name"),
                    rule.get("description"),
                    rule.get("lastUser"),
                    rule["lastDate"],
                    rule.get("tag"),
                    json.dumps(rule["lvl"])  # Convert the dictionary to JSON string
                ))
            conn.commit()
            print("Data inserted successfully.")


In [12]:
with open(data_dir / "distribution_rules.json") as file:
    data = json.load(file)

rules = data.get("rules", [])

for rule in rules:
    rrg = rule.get("lvl", {}).get("rrg")
    if rrg is None:
        rule["lvl"]["rrg"] = {"t": 0, "f": 0, "u": 0}

    last_date_str = rule.get("lastDate")
    if last_date_str:
        # Convert the existing date format
        last_date = datetime.strptime(last_date_str, "%d-%m-%Y %H:%M:%S")
        # Update the format to 'YYYY-MM-DD HH:MM:SS'
        rule["lastDate"] = last_date.strftime("%Y-%m-%d %H:%M:%S")

insert_data(rules)

Data inserted successfully.
